# Reading errors and debugging

Python stage, lesson P8. Part of [datafromzero](https://datafromzero.github.io), data and text CC BY 4.0.

**Goals:** read a traceback from the last line; name the cause of `SyntaxError`, `NameError`, `TypeError`, `ValueError` and `KeyError`; isolate a bug by printing intermediate values; handle an expected
failure with `try` and `except`.

**How to use this notebook:** run the cells from top to bottom. When in doubt, **Run all from the top**: restart the session and run every cell from the first.
The cells that fail on purpose run through a helper called `show_error`, which prints the error and lets the notebook go on, so Run all never stops.
Every exercise has a cell for your code and a check cell that compares your answer with a known answer from the cafe-sales data. Hints and solutions are folded away
under each exercise, free to open.

## Setup

In [ ]:
import linecache
import math
import numbers
import sys
import traceback

import numpy as np
import pandas as pd

# Run this cell as it is. It holds the answer checker and show_error, the helper that prints an error without stopping the notebook.
EXERCISES = ["Exercise 1", "Exercise 2", "Exercise 3", "Exercise 4"]


RESULTS = {}


def same(got, expected):
    """True when two values agree: floats within rounding, lists and dicts item by item."""
    try:
        if isinstance(expected, dict):
            return isinstance(got, dict) and got.keys() == expected.keys() and all(same(got[k], expected[k]) for k in expected)
        if isinstance(expected, (list, tuple)):
            return isinstance(got, (list, tuple)) and len(got) == len(expected) and all(same(g, e) for g, e in zip(got, expected))
        if isinstance(expected, bool) or isinstance(got, bool):
            return type(got) is type(expected) and got == expected
        if isinstance(expected, float):
            return isinstance(got, numbers.Real) and math.isclose(got, expected, rel_tol=1e-9, abs_tol=1e-9)
        if isinstance(expected, int):
            return isinstance(got, numbers.Integral) and got == expected
        return type(got) is type(expected) and got == expected
    except Exception:
        return False


def check(label, name, expected, traps=None):
    """Compare the variable called `name` with the known answer. It never stops Run all."""
    got = globals().get(name)
    if got is None:
        RESULTS[label] = None
        print(f"{label}: give {name} a value, then run this cell again.")
        return False
    if same(got, expected):
        RESULTS[label] = True
        print(f"{label}: correct.")
        return True
    RESULTS[label] = False
    print(f"{label}: not yet. Expected {expected!r}, but {name} is {got!r}.")
    for wrong, message in (traps or {}).items():
        if same(got, wrong):
            print("Hint:", message)
    return False

def check_function(label, name, cases):
    """Call your function on every case. Each case is (arguments, keyword arguments, expected result)."""
    fn = globals().get(name)
    if not callable(fn):
        RESULTS[label] = None
        print(f"{label}: define a function called {name}, then run this cell again.")
        return False
    for args, kwargs, expected in cases:
        try:
            got = fn(*args, **kwargs)
        except Exception as error:
            RESULTS[label] = False
            print(f"{label}: not yet. {name}{args} stopped with {type(error).__name__}: {error}")
            return False
        if not same(got, expected):
            RESULTS[label] = False
            print(f"{label}: not yet. {name}{args} should give {expected!r}, but gave {got!r}.")
            if got is None:
                print("Hint: None means the function gave nothing back. Did it return the value, or only print it?")
            return False
    RESULTS[label] = True
    print(f"{label}: correct ({len(cases)} cases).")
    return True

def show_error(code, short=False):
    """Run a few lines that fail on purpose and print the error, so Run all keeps going."""
    code = code.strip("\n") + "\n"
    linecache.cache["<cell>"] = (len(code), None, code.splitlines(True), "<cell>")
    try:
        exec(compile(code, "<cell>", "exec"), globals())
    except Exception as error:
        if short:
            print("".join(traceback.format_exception_only(type(error), error)).strip().split("\n")[-1])
        else:
            traceback.print_exception(type(error), error, error.__traceback__.tb_next, file=sys.stdout)

## Data

In [ ]:
# The five cafe-sales tables (version 1), read from the release tag v0.1.0.
BASE = "https://raw.githubusercontent.com/datafromzero/datafromzero.github.io/v0.1.0/datasets/cafe-sales/v1/"
stores = pd.read_csv(BASE + "stores.csv")
customers = pd.read_csv(BASE + "customers.csv")
products = pd.read_csv(BASE + "products.csv")
orders = pd.read_csv(BASE + "orders.csv")
order_items = pd.read_csv(BASE + "order_items.csv")

# Known answer from the dataset README: 3, 150, 16, 1,800 and 3,287 rows.
assert (len(stores), len(customers), len(products), len(orders), len(order_items)) == (3, 150, 16, 1800, 3287)
print("Data loaded: 3 stores, 150 customers, 16 products, 1,800 orders, 3,287 order lines.")

## Unit 1: Read the traceback from the last line

A report of the cell's failing call comes next. **Predict first.** Which kind of error will it end with? The quantity of the line arrives as the text `"2"`. Then run the cell and read the report from its last line.

In [ ]:
broken = """
def line_amount(quantity, unit_price, discount):
    return quantity * unit_price - discount


line_amount("2", 5.00, 0.50)
"""
show_error(broken)

### Exercise 1: A typo in a column name

**Task:** The cell below has a typo in a column name. Run it, read the report from its last line, and set `answer_1` to the name of the exception, as text.

**Expected:** one word in quotes, the name of an exception.

In [ ]:
# Exercise 1: run this cell, read the last line of the report, then set answer_1.
show_error('orders["order_idd"]')
answer_1 = None

In [ ]:
check("Exercise 1", "answer_1", "KeyError", traps={"KeyError: 'order_idd'": "Right line, but give only the name before the colon: the message after it is the detail."})

<details>
<summary>Hint</summary>

Ignore the middle of the report. The last line starts with the name of the error, then a colon, then the message.

</details>

<details>
<summary>Solution</summary>

```python
answer_1 = "KeyError"
```

The last line reads `KeyError: 'order_idd'`: the table has no column with that name.

</details>

## Unit 2: The five common errors

**Predict first.** Five lines, each failing for a different reason. Name the error of each before you run the cell. It prints only the last line of each report.

In [ ]:
show_error('print("Total:" 4.5)', short=True)
show_error('print(refund_total)', short=True)
show_error('"2" * 4.5', short=True)
show_error('int("two")', short=True)
show_error('stores["population"]', short=True)

## Unit 3: Isolate a bug with print

Order 1107 without discounts is 2 Mochas at 5.00 and 1 Cappuccino at 4.50: 14.5. **Predict first.** This function should return that. What does it return? The `print` lines show what happens inside.

In [ ]:
def order_total(prices, quantities):
    total = 0
    for price, quantity in zip(prices, quantities):
        line = price * quantity
        total = line
        print("line:", line, "total so far:", total)
    return total


order_total([5.0, 4.5], [2, 1])

## Unit 4: Handle an expected failure

**Predict first.** One of the three prices from the till export is the text `n/a`. What does the list hold at the end?

In [ ]:
raw_prices = ["4.50", "n/a", "3.25"]
prices = []
for text in raw_prices:
    try:
        prices.append(float(text))
    except ValueError:
        prices.append(None)
prices

## Unit 5: Check: name the error

**Predict first.** Name the error of each of these three lines, then run the cell.

In [ ]:
show_error('int("4.50")', short=True)
show_error('{"card": 421}["cash"]', short=True)
show_error('len(5)', short=True)

## Finish it

### Exercise 2: Finish it

**Task:** Three quantities came from the till export as text, and one of them, `"x"`, is a typo. The loop turns the readable ones into numbers, but its `except` block drops the bad one. Replace `pass` so that an unreadable quantity becomes 0.

**Expected:** a list of three whole numbers.

In [ ]:
raw_quantities = ["2", "x", "1"]
quantities = []
for text in raw_quantities:
    try:
        quantities.append(int(text))
    except ValueError:
        pass  # replace pass: add 0 for a quantity that is not a number
answer_2 = quantities

In [ ]:
check("Exercise 2", "answer_2", [2, 0, 1], traps={(2, 1): "pass drops the bad value and the list lost an item. Add a 0 in its place."})

<details>
<summary>Hint</summary>

Unit 4 appends `None` for a price it cannot read. Here the stand-in value is 0.

</details>

<details>
<summary>Solution</summary>

```python
except ValueError:
    quantities.append(0)
```

The answer is `[2, 0, 1]`.

</details>

## On your own

### Exercise 3: On your own

**Task:** The function below breaks on the first cell run: the quantity of a line arrives as text. Read the report from its last line, then change the function so that it also works when `quantity` is text, and still works when it is a number. Keep the name. Before you start, say in one sentence which line of the function the report points at. The check calls your function with text and with numbers.

**Expected:** `Exercise 3: correct (3 cases).`

In [ ]:
# Exercise 3: edit line_amount so that quantity may be text such as "2" or a number.
def line_amount(quantity, unit_price, discount):
    return quantity * unit_price - discount


show_error('line_amount("2", 5.0, 0.5)', short=True)

In [ ]:
check_function("Exercise 3", "line_amount", [(("2", 5.0, 0.5), {}, 9.5), ((2, 5.0, 0.5), {}, 9.5), (("3", 3.5, 0.0), {}, 10.5)])

<details>
<summary>Hint</summary>

The last line says Python cannot multiply the text by a decimal number. Lesson P2 has the conversion that fixes the text. Convert inside the function, on the line the report points at.

</details>

<details>
<summary>Solution</summary>

```python
def line_amount(quantity, unit_price, discount):
    return int(quantity) * unit_price - discount
```

The check prints `Exercise 3: correct (3 cases).` `int()` leaves a number unchanged, so the text and the number both work.

</details>

## Check yourself

### Exercise 4: Check yourself: spot the flaw

**Task:** A colleague wanted no more red errors, wrapped the loop in `try` and a bare `except`, and now the total reads 0. The total of the four prices that can be read should not be 0. Run the cell, then fix it: name the one error you expect, and find the typo that the `except` was hiding. Set `answer_4 = total`.

**Expected:** one decimal number.

In [ ]:
raw_prices = ["4.50", "n/a", "3.25", "4.75"]
total = 0
for text in raw_prices:
    try:
        total = totl + float(text)  # a typo hides in this line
    except:
        pass
print("total:", total)
answer_4 = None  # fix the code above, then set answer_4 = total

In [ ]:
check("Exercise 4", "answer_4", 12.5, traps={0: "total is 0 because every pass hit a NameError that the bare except swallowed. Name the error you expect, ValueError, and fix the typo."})

<details>
<summary>Hint</summary>

Take the `try` and `except` away for a moment and run the line on its own: what does the last line of the report say? Then put back an `except` that names only the error you expect.

</details>

<details>
<summary>Solution</summary>

```python
total = total + float(text)
...
except ValueError:
    pass
```

The answer is 12.5: 4.5 + 3.25 + 4.75. The typo `totl` raised a `NameError` on every pass, and the bare `except` hid all four.

</details>

## Run all from the top

Restart the session, run every cell from the first, then read this cell last.

In [ ]:
for label in EXERCISES:
    state = RESULTS.get(label)
    print(f"{label}: " + ("correct" if state else "not yet" if state is False else "not started"))
print(f"{sum(1 for label in EXERCISES if RESULTS.get(label))} of {len(EXERCISES)} exercises correct.")